In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter, ImageFont

# Retrieve Valid System Fonts

In [2]:
IMG_SIZE = 28
SCALE = 4 # smooth edges are created by rendering at bigger scale, then downsampling
CANVAS = IMG_SIZE * SCALE

FONT_EXTENSIONS = {".ttf", ".otf", ".ttc"}
SKIP_FONTS = ("dings", "symbol", "emoji", "icon", "math", "braille") # Fonts with symbols and emojis
FONT_DIRS = ["/System/Library/Fonts","/Library/Fonts","~/Library/Fonts"]

In [5]:
def find_fonts(directories: list[str]) -> list[Path]:
    """Return all font files found in the given directories."""
    found = set()
    for directory in directories:
        path = Path(directory).expanduser()
        if not path.is_dir():
            continue
        for file in path.rglob("*"):
            name = file.name.lower() # to eliminate SKIP_FONTS case sensitivity
            if file.suffix.lower() in FONT_EXTENSIONS and not any(k in name for k in SKIP_FONTS):
                found.add(file)
    return sorted(found)

def contains_digits(font_path: Path) -> bool:
    """Returns True if the font draws distinct, non-empty images for characters 1-9."""
    try:
        font = ImageFont.truetype(str(font_path), 64)
        shapes = set()
        for digit in "123456789":
            img = Image.new("L", (96, 96), 0) # create a 8-bit greyscale canvas
            ImageDraw.Draw(img).text((10, 10), digit, fill=255, font=font) # fill canvas with white digit
            if img.getbbox() is None: # if no digit is drawn, no bounding box will be found
                return False
            shapes.add(img.tobytes())
        return len(shapes) == 9
    except (OSError, ValueError):
        return False

def render_digits(font_path: Path) -> list[Image.Image]:
    """For debugging purposes: render 1-9 as images using matplotlib."""
    font = ImageFont.truetype(str(font_path), 64)
    images = []
    for digit in "123456789":
        img = Image.new("L", (96, 96), 0)
        ImageDraw.Draw(img).text((10, 10), digit, fill=255, font=font)
        images.append(img)

    fig, axes = plt.subplots(1, 9, figsize=(14, 2))
    for ax, img, digit in zip(axes, images, "123456789"):
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        ax.set_title(digit)
        ax.axis("off")
    fig.suptitle(f"{font_path.name}")
    plt.savefig(f"images/font_samples/{font_path.name}.png", dpi=300)
    plt.close()

Found 469 fonts. Begin processing...
Scanned 0 fonts. Rejected 0/469 fonts.
Scanned 46 fonts. Rejected 0/469 fonts.
Scanned 92 fonts. Rejected 1/469 fonts.
Scanned 138 fonts. Rejected 3/469 fonts.
Scanned 184 fonts. Rejected 16/469 fonts.
Scanned 230 fonts. Rejected 19/469 fonts.
Scanned 276 fonts. Rejected 26/469 fonts.
Scanned 322 fonts. Rejected 72/469 fonts.
Scanned 368 fonts. Rejected 117/469 fonts.
Scanned 414 fonts. Rejected 146/469 fonts.


/var/folders/03/q2hstsdj5vb6l8cy__5q8f_80000gn/T/ipykernel_94763/2819539423.py:44: UserWarning: Glyph 12498 (\N{KATAKANA LETTER HI}) missing from current font.
  plt.savefig(f"images/font_samples/{font_path.name}.png", dpi=300)
/var/folders/03/q2hstsdj5vb6l8cy__5q8f_80000gn/T/ipykernel_94763/2819539423.py:44: UserWarning: Glyph 12521 (\N{KATAKANA LETTER RA}) missing from current font.
  plt.savefig(f"images/font_samples/{font_path.name}.png", dpi=300)
/var/folders/03/q2hstsdj5vb6l8cy__5q8f_80000gn/T/ipykernel_94763/2819539423.py:44: UserWarning: Glyph 12461 (\N{KATAKANA LETTER KI}) missing from current font.
  plt.savefig(f"images/font_samples/{font_path.name}.png", dpi=300)
/var/folders/03/q2hstsdj5vb6l8cy__5q8f_80000gn/T/ipykernel_94763/2819539423.py:44: UserWarning: Glyph 12441 (\N{COMBINING KATAKANA-HIRAGANA VOICED SOUND MARK}) missing from current font.
  plt.savefig(f"images/font_samples/{font_path.name}.png", dpi=300)
/var/folders/03/q2hstsdj5vb6l8cy__5q8f_80000gn/T/ipykernel_94

Scanned 460 fonts. Rejected 150/469 fonts.


In [ ]:
# OPTIONAL DEBUG STEP: save image sets for each font in images/font_samples for manual validation

font_paths = find_fonts(FONT_DIRS)
print(f"Found {len(font_paths)} fonts. Begin processing...")
n_font, n_rejected = 1,0
percentile = len(font_paths)//10
for font in font_paths:
    if contains_digits(font):
        render_digits(font)
    else:
        n_rejected += 1
    if n_font%percentile == 0 or n_font == len(font_paths):
        print(f"Scanned {n_font} fonts. Rejected {n_rejected}/{len(font_paths)} fonts.")
    n_font += 1

In [ ]:
"""
Problematic fonts:
Bodoni Ornaments.ttf - symbols
NotoNastaliq.ttc - cutoff
Diwan Kufi.ttc - cutoff
DecoTypeNastaleeqUrdu.ttc - cutoff
Zapfino.ttf - cutoff
"""